# Neural Networks for Tabular Data — Exercises

These exercises accompany the lecture *"Neural Networks for Tabular Data"*
(multilayer perceptrons, backpropagation, training, initialization,
regularization). There are **5 short exercises**, each focused on one part
of the lecture:

1. **Perceptron vs. MLP** — the XOR problem and why hidden layers matter
2. **Backpropagation** — the VJP formulas from the lecture, applied by hand and checked numerically
3. **Vanishing gradients** — sigmoid vs. ReLU in a deep network
4. **Weight initialization** — naive vs. Xavier/Glorot vs. He
5. **Regularization** — weight decay, dropout, and early stopping

In every exercise, almost all of the code (data loading, model setup,
training loops, plotting) is **already written for you**. You only need to
fill in a few short lines marked `# TODO`, using the formulas given in the
recap just above each exercise. Run cells top to bottom.


## 0. Required libraries

Before running this notebook, make sure the following Python libraries are installed:

| Library | Used for |
|---|---|
| `numpy` | numerical arrays, linear algebra |
| `pandas` | (lightly used) data handling |
| `matplotlib` | plotting |
| `scikit-learn` | datasets, `Perceptron`, `MLPClassifier`, preprocessing |
| `torch` (PyTorch) | building/training neural networks in Exercises 3–5 |

If you don't have them yet, you can install everything with:

```bash
pip install numpy pandas matplotlib scikit-learn torch
```

(PyTorch install commands can vary by OS/GPU — see https://pytorch.org/get-started/locally/
if the plain `pip install torch` above doesn't work for you.)

The cell below **checks which of these libraries are already available**, and
tells you exactly what to install if something is missing.


In [ ]:
import importlib

# Maps the name used in `import ...` to the name used by `pip install ...`
# (they differ for scikit-learn: import sklearn, but pip install scikit-learn)
required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "sklearn": "scikit-learn",
    "torch": "torch",
}

missing = []
for import_name, pip_name in required_packages.items():
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown version")
        print(f"[OK]      {import_name:10s} is installed (version {version})")
    except ImportError:
        print(f"[MISSING] {import_name:10s} -- install it with: pip install {pip_name}")
        missing.append(pip_name)

print()
if missing:
    print("Some libraries are missing. Run this command in a terminal (or a notebook")
    print("cell prefixed with '!') to install all of them at once:\n")
    print(f"    pip install {' '.join(missing)}")
else:
    print("All required libraries are installed -- you're good to go!")


## Imports

Common imports used throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.linear_model import Perceptron
from sklearn.neural_network import MLPClassifier
from sklearn.datasets import load_breast_cancer, load_wine, make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(0)
torch.manual_seed(0)


## Exercise 1 — The XOR problem: Perceptron vs. MLP

**Recap.** A single perceptron computes $f(\mathbf{x}) = H(\mathbf{w}^\top\mathbf{x} + b)$,
a *linear* decision boundary. The XOR function

| $x_1$ | $x_2$ | $y$ |
|---:|---:|---:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

is the classic example that is **not linearly separable**, so a single
perceptron cannot solve it. Stacking perceptrons into a multilayer perceptron
(MLP) with at least one hidden layer fixes this.

**Task.** The dataset and the plotting helper are given. You just need to
write **2 lines**:
1. Fit a `Perceptron` on `(X, y)`.
2. Fit an `MLPClassifier` with one hidden layer of 4 units and a nonlinear
   activation (e.g. `activation='tanh'`) on `(X, y)`.

Then compare the two accuracies and decision boundaries (already plotted for you).


In [ ]:
rng = np.random.RandomState(0)

# --- XOR dataset (given) ---
X_base = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_base = np.array([0, 1, 1, 0])
n_repeats = 50
X = np.repeat(X_base, n_repeats, axis=0) + rng.normal(0, 0.15, size=(n_repeats * 4, 2))
y = np.repeat(y_base, n_repeats)

# --- TODO (2 lines): fit a Perceptron, and an MLPClassifier(hidden_layer_sizes=(4,), activation='tanh') ---
perceptron = None
mlp = None

acc_perceptron = perceptron.score(X, y)
acc_mlp = mlp.score(X, y)
print("Perceptron accuracy on XOR:", acc_perceptron)
print("MLP accuracy on XOR:", acc_mlp)

# --- plotting (given) ---
def plot_decision_boundary(model, X, y, title, ax):
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k')
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
plot_decision_boundary(perceptron, X, y, f"Perceptron (acc={acc_perceptron:.2f})", axes[0])
plot_decision_boundary(mlp, X, y, f"MLP (acc={acc_mlp:.2f})", axes[1])
plt.tight_layout()
plt.show()


## Exercise 2 — Backpropagation: filling in the VJP formulas

**Recap.** For a one-hidden-layer MLP, $\hat{\mathbf y} = \mathbf{W}_2\,\varphi(\mathbf{W}_1\mathbf{x}+\mathbf{b}_1)+\mathbf{b}_2$,
with squared-error loss, the lecture derived these VJP rules:
* **Linear layer** $\mathbf z=\mathbf W\mathbf x$: VJP wrt input is $\mathbf u^\top\mathbf W$; VJP wrt weights is the outer product $\mathbf u\mathbf x^\top$.
* **Elementwise nonlinearity** $\mathbf z=\varphi(\mathbf x)$: VJP is $\mathbf u\odot\varphi'(\mathbf x)$ — for ReLU, $\varphi'(a)=\mathbb I(a>0)$.

**Task.** Everything below — `forward()`, the gradient checker, and the
training loop — is already implemented. **The only thing missing is 4 lines
inside `backward()`**: the gradients `dW2`, `dZ1`, `dA1`, `dW1`. Use the
formulas in the comments (they are exactly the rules above). Once filled in,
run the gradient check — the relative error should be tiny (~1e-7 or
smaller) — and watch the training loss decrease.


In [ ]:
class SimpleMLP:
    """One-hidden-layer MLP: Linear -> ReLU -> Linear, squared-error loss."""

    def __init__(self, n_in, n_hidden, n_out, seed=0):
        rng = np.random.RandomState(seed)
        self.W1 = rng.randn(n_in, n_hidden) * np.sqrt(2.0 / n_in)   # He init
        self.b1 = np.zeros(n_hidden)
        self.W2 = rng.randn(n_hidden, n_out) * np.sqrt(2.0 / n_hidden)
        self.b2 = np.zeros(n_out)

    def forward(self, X):
        self.X = X
        self.A1 = X @ self.W1 + self.b1            # pre-activation, (N, H)
        self.Z1 = np.maximum(self.A1, 0)             # ReLU,            (N, H)
        self.Y_hat = self.Z1 @ self.W2 + self.b2     # output,          (N, O)
        return self.Y_hat

    def backward(self, y):
        N = self.X.shape[0]
        dY_hat = (self.Y_hat - y) / N   # dL/dY_hat, (N, O) -- given

        # TODO (4 lines): fill in the 4 gradients below.
        #   dW2 = Z1.T @ dY_hat          (outer product: linear layer wrt weights)
        #   dZ1 = dY_hat @ W2.T          (linear layer wrt input)
        #   dA1 = dZ1 * (A1 > 0)         (elementwise ReLU derivative)
        #   dW1 = X.T @ dA1              (outer product: linear layer wrt weights)
        dW2 = None
        db2 = dY_hat.sum(axis=0)
        dZ1 = None
        dA1 = None
        dW1 = None
        db1 = dA1.sum(axis=0)

        self.grads = dict(W1=dW1, b1=db1, W2=dW2, b2=db2)
        return self.grads

    def loss(self, X, y):
        y_hat = self.forward(X)
        return 0.5 * np.mean((y_hat - y) ** 2)

    def params(self):
        return dict(W1=self.W1, b1=self.b1, W2=self.W2, b2=self.b2)

    def step(self, lr):
        self.W1 -= lr * self.grads['W1']
        self.b1 -= lr * self.grads['b1']
        self.W2 -= lr * self.grads['W2']
        self.b2 -= lr * self.grads['b2']


# --- numerical gradient checker (given) ---
def numerical_gradient_check(model, X, y, eps=1e-5):
    model.forward(X)
    model.backward(y)
    analytical = model.grads
    max_rel_err = 0.0
    for name, param in model.params().items():
        grad_analytical = analytical[name]
        grad_numerical = np.zeros_like(param)
        it = np.nditer(param, flags=['multi_index'])
        for _ in it:
            idx = it.multi_index
            orig = param[idx]
            param[idx] = orig + eps
            loss_plus = model.loss(X, y)
            param[idx] = orig - eps
            loss_minus = model.loss(X, y)
            param[idx] = orig
            grad_numerical[idx] = (loss_plus - loss_minus) / (2 * eps)
        rel_err = np.abs(grad_analytical - grad_numerical) / (
            np.abs(grad_analytical) + np.abs(grad_numerical) + 1e-8)
        max_rel_err = max(max_rel_err, rel_err.max())
        print(f"  {name}: max relative error = {rel_err.max():.2e}")
    return max_rel_err


# --- gradient check on a tiny random problem (given) ---
rng = np.random.RandomState(1)
N, D, H, O = 6, 3, 5, 1
X_small = rng.randn(N, D)
y_small = rng.randn(N, O)

model = SimpleMLP(D, H, O, seed=1)
print("Gradient check (should be ~1e-7 or smaller):")
max_err = numerical_gradient_check(model, X_small, y_small)
print(f"Max relative error: {max_err:.2e}")

# --- train on a small synthetic regression dataset (given) ---
Xr, yr = make_regression(n_samples=200, n_features=3, noise=5.0, random_state=0)
yr = yr.reshape(-1, 1)
Xr = (Xr - Xr.mean(0)) / Xr.std(0)
yr = (yr - yr.mean()) / yr.std()

model2 = SimpleMLP(3, 16, 1, seed=2)
losses = []
for it in range(500):
    model2.forward(Xr)
    model2.backward(yr)
    model2.step(lr=0.5)
    losses.append(model2.loss(Xr, yr))

print("Initial loss:", losses[0])
print("Final loss:", losses[-1])

plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("training loss")
plt.title("Training SimpleMLP with hand-written backprop")
plt.show()


## Exercise 3 — Vanishing gradients: Sigmoid vs. ReLU

**Recap.** Sigmoid units saturate for large $|a|$, where $\varphi'(a)=\sigma(a)(1-\sigma(a))\approx 0$.
In a deep network the chain rule multiplies many such small derivatives
together, so the gradient reaching early layers can vanish. ReLU has
derivative $1$ for positive inputs and does not saturate on that side, which
is why it is the modern default.

**Task.** Below, two deep (8-layer) MLPs are built and trained for one step
on `load_breast_cancer` — one with `Sigmoid`, one with `ReLU` — and the
gradient norm of every layer's weights is computed and plotted for you
(everything is already implemented).

**The only thing to add (2 lines):** compute the ratio of the **first**
layer's gradient norm to the **last** layer's gradient norm, for each
network, and print them. Which ratio is closer to 0 (i.e. shows more
vanishing)?


In [ ]:
data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data)
y = data.target
X_t = torch.tensor(X, dtype=torch.float32)
y_t = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

n_features = X.shape[1]
n_hidden_layers = 8
hidden_size = 32

# --- network builder (given) ---
def build_deep_mlp(activation_cls):
    layers = []
    in_dim = n_features
    for _ in range(n_hidden_layers):
        layers.append(nn.Linear(in_dim, hidden_size))
        layers.append(activation_cls())
        in_dim = hidden_size
    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers)

# --- one forward+backward pass, collecting gradient norms (given) ---
def gradient_norms_per_layer(model, X, y):
    criterion = nn.BCEWithLogitsLoss()
    model.zero_grad()
    loss = criterion(model(X), y)
    loss.backward()
    return [layer.weight.grad.norm().item() for layer in model if isinstance(layer, nn.Linear)]

model_sigmoid = build_deep_mlp(nn.Sigmoid)
model_relu = build_deep_mlp(nn.ReLU)
norms_sigmoid = gradient_norms_per_layer(model_sigmoid, X_t, y_t)
norms_relu = gradient_norms_per_layer(model_relu, X_t, y_t)
print("Sigmoid gradient norms per layer:", norms_sigmoid)
print("ReLU gradient norms per layer:", norms_relu)

plt.figure(figsize=(6, 4))
plt.semilogy(range(1, n_hidden_layers + 2), norms_sigmoid, marker='o', label='Sigmoid')
plt.semilogy(range(1, n_hidden_layers + 2), norms_relu, marker='s', label='ReLU')
plt.xlabel('Layer (1 = closest to input)')
plt.ylabel('Gradient norm (log scale)')
plt.title('Vanishing gradients: Sigmoid vs ReLU')
plt.legend()
plt.show()

# --- TODO (2 lines): ratio of first-layer to last-layer gradient norm ---
# hint: ratio = norms[0] / norms[-1]
ratio_sigmoid = None
ratio_relu = None
print(f"first/last layer grad-norm ratio -- sigmoid: {ratio_sigmoid:.2e}, relu: {ratio_relu:.2e}")


## Exercise 4 — Weight initialization: naive vs. Xavier vs. He

**Recap.** To keep activation variance stable across layers, weight variance
should scale roughly as $\sigma^2\propto 1/n_\text{in}$. This gives:
* **Xavier/Glorot**: $\sigma^2 = 2/(n_\text{in}+n_\text{out})$ — good default for tanh/sigmoid/linear units.
* **He**: $\sigma^2 = 2/n_\text{in}$ — recommended for ReLU units (`nn.init.kaiming_uniform_(..., nonlinearity='relu')`).

A too-large ("naive") initialization variance can make activations (and
gradients) explode in a deep network.

**Task.** A 10-layer ReLU network, the training loop, and the comparison
plot are all given below, along with the `'naive'` and `'xavier'`
initialization branches. **Add only the missing `'he'` branch (1 line)**
using `nn.init.kaiming_uniform_`.


In [ ]:
data = load_wine()
X = StandardScaler().fit_transform(data.data)
y = data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

n_features, n_classes = X.shape[1], len(np.unique(y))
n_hidden_layers, hidden_size = 10, 64

# --- network builder (given) ---
def build_mlp():
    layers = []
    in_dim = n_features
    for _ in range(n_hidden_layers):
        layers.append(nn.Linear(in_dim, hidden_size))
        layers.append(nn.ReLU())
        in_dim = hidden_size
    layers.append(nn.Linear(in_dim, n_classes))
    return nn.Sequential(*layers)

def apply_init(model, method):
    for layer in model:
        if isinstance(layer, nn.Linear):
            if method == 'naive':
                nn.init.normal_(layer.weight, mean=0.0, std=1.0)
            elif method == 'xavier':
                nn.init.xavier_uniform_(layer.weight)
            elif method == 'he':
                pass  # TODO (1 line): He/Kaiming init, recommended default for ReLU
            nn.init.zeros_(layer.bias)

# --- training loop (given) ---
def train(model, n_epochs=100, lr=0.01):
    optimizer = optim.SGD(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(n_epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses

results = {}
for method in ['naive', 'xavier', 'he']:
    model = build_mlp()
    apply_init(model, method)
    losses = train(model)
    with torch.no_grad():
        test_acc = (model(X_test_t).argmax(1) == y_test_t).float().mean().item()
    results[method] = (losses, test_acc)
    print(f"{method:8s} init -- final train loss: {losses[-1]:.4f}, test accuracy: {test_acc:.3f}")

plt.figure(figsize=(6, 4))
for method, (losses, acc) in results.items():
    plt.plot(losses, label=f"{method} (test acc={acc:.2f})")
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("Effect of weight initialization on training speed")
plt.legend()
plt.show()


## Exercise 5 — Regularization: weight decay, dropout, early stopping

**Recap.**
* **Weight decay** — an $\ell_2$ penalty on the weights (`weight_decay=...` in the optimizer).
* **Dropout** — randomly zero a unit's outputs with probability $p$ during training (`nn.Dropout(p)`).
* **Early stopping** — stop once the validation loss stops improving.

**Task.** We deliberately use a **small training set** (80 examples) from
`load_breast_cancer` so the network overfits. The model (with an optional
dropout layer) and the full training loop are already written, including the
early-stopping bookkeeping. **The only thing missing is 1 line**: the
boolean condition that decides whether the validation loss has *improved*
this epoch (hint: compare `val_loss` to `best_val`, allowing a small
tolerance of `1e-4`). Then compare the 4 train/validation curves.


In [ ]:
data = load_breast_cancer()
X = StandardScaler().fit_transform(data.data)
y = data.target
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=80, random_state=0, stratify=y)
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).reshape(-1, 1)
n_features = X.shape[1]

# --- model with optional dropout (given) ---
def build_model(use_dropout=False, p=0.5):
    layers = [nn.Linear(n_features, 64), nn.ReLU()]
    if use_dropout:
        layers.append(nn.Dropout(p))
    layers += [nn.Linear(64, 64), nn.ReLU()]
    if use_dropout:
        layers.append(nn.Dropout(p))
    layers.append(nn.Linear(64, 1))
    return nn.Sequential(*layers)

# --- training loop with early stopping (given, except 1 line) ---
def train_with_history(model, weight_decay=0.0, n_epochs=300, lr=1e-3,
                        early_stopping=False, patience=20):
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    train_losses, val_losses = [], []
    best_val, best_state, epochs_no_improve, stopped_epoch = np.inf, None, 0, n_epochs

    for epoch in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train_t), y_train_t)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()
        val_losses.append(val_loss)

        if early_stopping:
            # TODO (1 line): has val_loss improved on best_val by more than 1e-4?
            improved = None
            if improved:
                best_val = val_loss
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                epochs_no_improve = 0
            else:
                epochs_no_improve += 1
                if epochs_no_improve >= patience:
                    stopped_epoch = epoch + 1
                    break

    if early_stopping and best_state is not None:
        model.load_state_dict(best_state)
    return train_losses, val_losses, stopped_epoch

def val_accuracy(model):
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X_val_t)) > 0.5).float()
        return (preds == y_val_t).float().mean().item()

# --- run the 4 configurations (given) ---
configs = {
    'baseline (no regularization)': dict(use_dropout=False, weight_decay=0.0, early_stopping=False),
    'weight decay (L2)':            dict(use_dropout=False, weight_decay=1e-2, early_stopping=False),
    'dropout':                      dict(use_dropout=True,  weight_decay=0.0, early_stopping=False),
    'early stopping':               dict(use_dropout=False, weight_decay=0.0, early_stopping=True),
}

results = {}
for name, cfg in configs.items():
    model = build_model(use_dropout=cfg['use_dropout'])
    tl, vl, stopped = train_with_history(model, weight_decay=cfg['weight_decay'],
                                          early_stopping=cfg['early_stopping'])
    acc = val_accuracy(model)
    results[name] = (tl, vl, stopped, acc)
    print(f"{name:30s} final train loss={tl[-1]:.4f}  final val loss={vl[-1]:.4f}  "
          f"stopped@{stopped}  val acc={acc:.3f}")

fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharey=True)
for ax, (name, (tl, vl, stopped, acc)) in zip(axes.ravel(), results.items()):
    ax.plot(tl, label='train')
    ax.plot(vl, label='val')
    ax.set_title(f"{name}\n(val acc={acc:.2f})")
    ax.set_xlabel('epoch')
    ax.legend()
plt.tight_layout()
plt.show()


## Wrap-up

You've now touched the core building blocks from the lecture: why hidden
layers matter (XOR), the VJP rules behind backpropagation, why activation
choice affects gradient flow, why initialization matters, and a few standard
ways to regularize a network — each with just a few lines of code of your
own, built on top of working infrastructure.
